Let's repeat what we did in the previous notebook (import libraries and clean the data).

In [15]:
#Importing necessary libraries 
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as mpl
import seaborn as sns

#Printing library versions for user clarification
print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")
print(f"seaborn version: {sns.__version__}")

data = pd.read_csv("../data/players.csv")
gk_columns = data.columns[range(54,72,1)]
data = data.drop(columns=gk_columns)
data = data.dropna(subset=["goals", "minutes", "assists"])
data = data.drop(columns=["pens_won", "pens_conceded"])

#Keep only numeric data
data = data.select_dtypes(include='number')

NaN_columns = ["plus_minus_wowy",
                "minutes_per_start",
                "games_complete", 
                "games_subs", 
                "minutes_per_sub", 
                "unused_subs", 
                "goals_per_shot", 
                "goals_per_shot_on_target",
                "shots_on_target_pct"]

data = data.drop(columns=NaN_columns)

NumPy version: 2.5.1
pandas version: 3.0.5
matplotlib version: 3.11.1
seaborn version: 0.13.2


We also need to import the necessary libraries for our ML models.

In [16]:
import sklearn
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [17]:
data.head()

,age,birth_year,games,games_starts,minutes,minutes_90s,goals,assists,goals_assists,goals_pens,...,plus_minus,plus_minus_per90,cards_yellow_red,fouls,fouled,offsides,crosses,interceptions,tackles_won,own_goals
1,23,2003,2,0,19.0,0.2,0.0,0.0,0.0,0.0,...,0.0,0.00,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,26,2000,4,3,251.0,2.8,1.0,0.0,1.0,1.0,...,-1.0,-0.36,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0
3,24,2002,3,1,98.0,1.1,0.0,0.0,0.0,0.0,...,-1.0,-0.92,0.0,0.0,2.0,0.0,5.0,1.0,0.0,0.0
4,34,1991,4,4,360.0,4.0,0.0,0.0,0.0,0.0,...,-4.0,-1.00,0.0,0.0,5.0,0.0,0.0,4.0,3.0,0.0
5,23,2002,4,4,360.0,4.0,0.0,0.0,0.0,0.0,...,-4.0,-1.00,0.0,3.0,2.0,1.0,6.0,3.0,5.0,0.0


We are going to use three different models and compare them to each other to see which will most accurately predict the number of goals scored by a player. 

1. Linear Regressor - We model the number of goals as a linear function of the other features 
2. Random Forest Regressor - We use a combination of different decision trees to predict number of goals
3. Ridge Regressor - We model the number of goals as a function of the other features and add a penalty term to shrink coefficients to zero
 
To do this, we split our dataset into training and validation sets, and compare the accuracy which will be calculated after cross-validation to see how effective our models are.

First, let's define our features and target variable. Since it would be quite easy to predict the number of goals given the number of assists and the number of goals+assists, we should remove all columns which encompass the number of goals in them so we can actually take some meaningful insight from our models.

In [18]:
#Create a mask which is true for all columns which contain the word "goals" 
#and then apply it to the dataset, keeping only columns which are false in the mask
X = data.loc[:, ~data.columns.str.contains("goals", case=False)]
X.head()

,age,birth_year,games,games_starts,minutes,minutes_90s,assists,pens_made,pens_att,cards_yellow,...,points_per_game,plus_minus,plus_minus_per90,cards_yellow_red,fouls,fouled,offsides,crosses,interceptions,tackles_won
1,23,2003,2,0,19.0,0.2,0.0,0.0,0.0,0.0,...,0.0,0.0,0.00,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,26,2000,4,3,251.0,2.8,0.0,0.0,0.0,0.0,...,1.0,-1.0,-0.36,0.0,1.0,1.0,0.0,1.0,0.0,1.0
3,24,2002,3,1,98.0,1.1,0.0,0.0,0.0,0.0,...,1.0,-1.0,-0.92,0.0,0.0,2.0,0.0,5.0,1.0,0.0
4,34,1991,4,4,360.0,4.0,0.0,0.0,0.0,0.0,...,1.0,-4.0,-1.00,0.0,0.0,5.0,0.0,0.0,4.0,3.0
5,23,2002,4,4,360.0,4.0,0.0,0.0,0.0,1.0,...,1.0,-4.0,-1.00,0.0,3.0,2.0,1.0,6.0,3.0,5.0


In [19]:
y = data.goals
y

1       0.0
2       1.0
3       0.0
4       0.0
5       0.0
       ... 
1242    0.0
1243    0.0
1244    0.0
1245    0.0
1247    0.0
Name: goals, Length: 1039, dtype: float64

Now let's split our dataset into training and testing data.

In [20]:
# Fix a seed for reproducibility
np.random.seed(13)

# Split into train & test set
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2) # 20% for test set

X_train.head()



,age,birth_year,games,games_starts,minutes,minutes_90s,assists,pens_made,pens_att,cards_yellow,...,points_per_game,plus_minus,plus_minus_per90,cards_yellow_red,fouls,fouled,offsides,crosses,interceptions,tackles_won
613,25,2000,1,0,24.0,0.3,0.0,0.0,0.0,0.0,...,0.00,-2.0,-7.50,0.0,1.0,0.0,0.0,0.0,1.0,0.0
130,23,2002,3,3,270.0,3.0,0.0,0.0,0.0,1.0,...,0.33,-5.0,-1.67,0.0,6.0,3.0,0.0,0.0,1.0,7.0
128,29,1997,5,5,474.0,5.3,0.0,1.0,1.0,0.0,...,2.20,6.0,1.14,0.0,7.0,10.0,1.0,15.0,2.0,6.0
1167,30,1996,1,0,7.0,0.1,0.0,0.0,0.0,0.0,...,3.00,1.0,12.86,0.0,0.0,0.0,0.0,0.0,0.0,0.0
646,24,2001,1,0,12.0,0.1,0.0,0.0,0.0,0.0,...,3.00,1.0,7.50,0.0,0.0,0.0,0.0,0.0,0.0,1.0


We can now train and evaluate our models. For an initial idea of how our models are performing, we will simply train the models on the training data and use the model to make predictions and compare with our testing data. We will then have three different metrics to base the accuracy of the model on:

1. MAE - The average error in our predications
2. MSE - The average square error in our predictions (Larger errors are emphasised more)
3. R^2 - The fraction of the variance in the number of goals explained by our model

In [21]:
import sys
sys.path.append("..")
from src.model_evaluation import train_and_evaluate

models = {
    "Linear Regression" : LinearRegression(),
    "Random Forest Regression" : RandomForestRegressor(random_state=13),
    "Ridge Regression" : Ridge(random_state=13)
}

model_scores = train_and_evaluate(models, 
                                  X_train, 
                                  y_train, 
                                  X_test, 
                                  y_test)

results_df = pd.DataFrame(model_scores).T  # models as rows, metrics as columns
print(results_df)

                               MSE       MAE       R^2
Linear Regression         0.123883  0.207629  0.829266
Random Forest Regression  0.235380  0.194327  0.675603
Ridge Regression          0.124667  0.207508  0.828185
